In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [2]:
df = pd.read_csv("final_data.csv")
df.head()

C:\Users\RS Computers\AppData\Local\Temp\ipykernel_8164\2446631114.py:1: DtypeWarning: Columns (1,2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("final_data.csv")


,performance_score,offensive_skill,defensive_skill,efficiency,aggression,mistake_penalty,experience,physical_index,game_type,player_name,Selected
0,-0.124196,-0.8277993019909841,-0.8246205859249759,-0.923196,-0.9200844481915191,0.865945,0.0,0.0,Cricket,Player_0,1
1,-0.077223,-0.8277993019909841,0.5490061582842831,-0.352970,-0.7651858259817703,-0.633086,0.0,0.0,Cricket,Player_1,1
2,-1.533367,-1.189937080634515,0.8924128443365978,-0.630985,-0.7083896645048627,-2.506875,0.0,0.0,Cricket,Player_2,0
3,-1.549024,-1.4313622663968693,-2.5416540161865497,-1.833758,-1.808169882194077,1.240703,0.0,0.0,Cricket,Player_3,0
4,-0.108538,-0.344948930466276,0.8924128443365978,-0.640333,-0.9097578733775357,-1.007844,0.0,0.0,Cricket,Player_4,1


In [3]:
# X = df.drop(columns=["player_name","Selected"])
# y = df["Selected"]

X = df.drop(["player_name", "Selected"], axis=1)

y = df["Selected"]

In [4]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

X["game_type"] = encoder.fit_transform(X["game_type"])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
X_train.dtypes

performance_score    float64
offensive_skill       object
defensive_skill       object
efficiency           float64
aggression            object
mistake_penalty      float64
experience           float64
physical_index       float64
game_type              int64
dtype: object

In [7]:
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

In [8]:
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [9]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
X = X.replace(r"[^\d.-]", "", regex=True)

In [11]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = model.predict(X_test)

In [13]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 1.0


In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19078
           1       1.00      1.00      1.00     12629

    accuracy                           1.00     31707
   macro avg       1.00      1.00      1.00     31707
weighted avg       1.00      1.00      1.00     31707



In [15]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[19078     0]
 [    0 12629]]


In [16]:
importance = pd.DataFrame({
    "Feature": df.drop(columns=["player_name","Selected"]).columns,
    "Importance": model.feature_importances_
})

importance.sort_values(by="Importance", ascending=False)

,Feature,Importance
0,performance_score,0.719468
2,defensive_skill,0.093625
4,aggression,0.084883
6,experience,0.037542
3,efficiency,0.027275
8,game_type,0.016829
1,offensive_skill,0.011456
7,physical_index,0.007552
5,mistake_penalty,0.001371


In [17]:
def predict_player(player_name, sport):

    # player search
    player = df[
        (df["player_name"] == player_name) &
        (df["game_type"] == sport)
    ]

    if player.empty:
        print("Player not found")
        return

    # features separate
    player_features = player.drop(columns=["player_name","Selected"])

    # encode sport
    player_features["game_type"] = encoder.transform(player_features["game_type"])

    # numeric convert
    player_features = player_features.apply(pd.to_numeric, errors="coerce").fillna(0)

    # scaling
    player_scaled = scaler.transform(player_features)

    # prediction
    prediction = model.predict(player_scaled)

    if prediction[0] == 1:
        print("Selected")
    else:
        print("Not Selected")

In [18]:
predict_player("Player_255","Cricket")

Selected


# Top 5 Player Recommendation

In [19]:
def top_players_by_sport(sport, n=5):

    sport_df = df[df["game_type"] == sport]

    top_players = sport_df.sort_values(
        by="performance_score",
        ascending=False
    ).head(n)

    return top_players[["player_name","performance_score"]]

In [20]:
top_players_by_sport("Football")

,player_name,performance_score
309,Player_309,3.757794
310,Player_310,3.625373
311,Player_311,3.492953
312,Player_312,3.492953
313,Player_313,3.492953


# Feature Importance Graph

In [22]:
feature_names = df.drop(columns=["player_name", "Selected", "performance_score"]).columns

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

ValueError: All arrays must be of the same length

In [ ]:
plt.figure(figsize=(8,5))

plt.barh(
    importance["Feature"],
    importance["Importance"]
)

plt.gca().invert_yaxis()

plt.title("Feature Importance for Player Selection")

plt.show()

# Model Comparision

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=200),
    "KNN": KNeighborsClassifier()
}

In [ ]:
results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    results[name] = accuracy

    print(name, "Accuracy:", accuracy)

In [ ]:
comparison = pd.DataFrame(
    list(results.items()),
    columns=["Model","Accuracy"]
)

comparison

In [ ]:
plt.figure(figsize=(7,5))

plt.bar(
    comparison["Model"],
    comparison["Accuracy"]
)

plt.title("Model Comparison")

plt.ylabel("Accuracy")

plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Selected","Selected"],
    yticklabels=["Not Selected","Selected"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()